# Functions, Files, and Errors

Write reusable data helpers, read and write files safely, and handle expected failures.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Define small functions with parameters and return values.
- Read and write text/CSV data with context managers.
- Catch specific exceptions without hiding programming mistakes.

A useful data helper performs one job, names its inputs clearly, and returns a value instead of only printing it.

In [1]:
def percent_change(old_value, new_value):
    """Return percentage change from old_value to new_value."""
    if old_value == 0:
        raise ValueError("old_value must not be zero")
    return (new_value - old_value) / old_value * 100

print(round(percent_change(80, 92), 1), "%")

try:
    percent_change(0, 5)
except ValueError as error:
    print("Handled expected problem:", error)

15.0 %
Handled expected problem: old_value must not be zero


## Default arguments and mutable defaults
Avoid using a mutable object such as `[]` as a default argument. It is created once when the function is defined and can accidentally be shared across calls.

In [2]:
# Safer pattern

def add_label(label, labels=None):
    if labels is None:
        labels = []
    labels.append(label)
    return labels

print(add_label("clean"))
print(add_label("review"))

# Type conversion should be deliberate.
def parse_count(text):
    try:
        return int(text)
    except (TypeError, ValueError):
        return None

print(parse_count("42"), parse_count("unknown"), parse_count(None))

['clean']
['review']
42 None None


## File paths and context managers
`with open(...)` closes a file even when an error occurs. For repeatable examples, temporary folders avoid leaving accidental files in the repository.

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv

rows = [
    {"city": "Baku", "visits": 12},
    {"city": "Ganja", "visits": 8},
]
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "visits.csv"
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["city", "visits"])
        writer.writeheader()
        writer.writerows(rows)
    with path.open("r", newline="", encoding="utf-8") as file:
        loaded = list(csv.DictReader(file))
    print(loaded)

[{'city': 'Baku', 'visits': '12'}, {'city': 'Ganja', 'visits': '8'}]


## Debugging habits
- Catch only the errors you know how to handle.
- Do not use a bare `except:` to silently continue.
- Print or inspect shapes, columns, types, and a few records before investigating complex failures.
- Keep raw inputs unchanged when a transformation is hard to reproduce.

In [4]:
raw_values = ["10", "12", "n/a", "15"]
converted = []
for raw in raw_values:
    try:
        converted.append(float(raw))
    except ValueError:
        converted.append(None)
print(converted)

[10.0, 12.0, None, 15.0]


## Practice
Write a function that converts a string to a float and returns `None` for invalid input. Do not catch unrelated exceptions.

In [5]:
def safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

print(safe_float("3.5"), safe_float("three"), safe_float(None))

3.5 None None
